# NB-00: Setup e Verificação de Ambiente

Verifica versões, configura paths globais e testa todas as APIs.

In [ ]:
import sys, json, importlib
from pathlib import Path
import subprocess

PROJECT_ROOT = Path(".").resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
RANDOM_SEED = 42
ALPHA = 0.95

print(f"Python: {sys.version}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

## Verificação de Dependências

In [ ]:
REQUIRED = ["torch", "gymnasium", "d3rlpy", "yfinance", "pandas",
            "numpy", "scipy", "matplotlib", "seaborn", "ipywidgets", "bcb"]

status = {}
for pkg in REQUIRED:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "?")
        status[pkg] = {"installed": True, "version": ver}
        print(f"  OK  {pkg}=={ver}")
    except ImportError:
        status[pkg] = {"installed": False, "version": None}
        print(f"  MISSING  {pkg}")

(OUTPUT_DIR / "models").mkdir(parents=True, exist_ok=True)
with open(OUTPUT_DIR / "models" / "environment_check.json", "w") as f:
    json.dump(status, f, indent=2)
print("
Saved: outputs/models/environment_check.json")

## Teste de APIs

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Test BCB SGS
try:
    from bcb import sgs
    s = sgs.get(432, start="2024-01-01", end="2024-01-31")
    print(f"BCB SGS OK — SELIC sample: {s.iloc[-1]:.2f}%")
except Exception as e:
    print(f"BCB SGS FAILED: {e}")

# Test yfinance
try:
    import yfinance as yf
    t = yf.Ticker("PETR4.SA")
    h = t.history(period="5d")
    print(f"yfinance OK — PETR4.SA last close: {h["Close"].iloc[-1]:.2f}")
except Exception as e:
    print(f"yfinance FAILED: {e}")

# Test Semantic Scholar
import requests
ss_key = os.getenv("SEMANTIC_SCHOLAR_API_KEY", "")
headers = {"x-api-key": ss_key} if ss_key else {}
r = requests.get("https://api.semanticscholar.org/graph/v1/paper/search",
                 params={"query": "offline reinforcement learning", "limit": 1},
                 headers=headers, timeout=10)
print(f"Semantic Scholar: {r.status_code}")